# A Neural Pipeline on 61 African Novels

**Computational Text Analysis Pack · Module 13 — Case Studies**

*Technique:* Reading a real project: architecture, corpus, and what it claimed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/16-case-studies/13a_the_case.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Modules 1–12 taught methods. This module takes one real project and asks the
question a peer reviewer would: **does the number mean what the author thinks it
means?**

The case is a small Keras pipeline for authorship attribution over 61 novels by
seven African novelists — Achebe, Okri, Gordimer, Ngũgĩ, Emecheta, Forna,
Tutuola. Six notebooks, one per step: install, load and split, vectorise, build
model, train, evaluate. The repository is public:

<https://github.com/Sagaustus/cnn_dh_practice>

It is a good project. It also contained a mistake that inflated its headline
number by 24 points, and finding that mistake is the lesson.

## The tooling

**Keras** (on TensorFlow) for the model — `Embedding → GlobalAveragePooling1D →
Dense`, about 32,000 parameters. **scikit-learn** for the split and the metrics.

Worth saying plainly because it is widely misunderstood: **this model needs no
GPU.** It trains in 8–17 seconds on a CPU. GPUs matter for transformers and
image models, not for a model this size. If someone tells you a neural method
requires a GPU, ask how many parameters.

## The architecture, and what it can and cannot see

```
Embedding(vocab=1000, dim=32)      each word becomes 32 numbers
GlobalAveragePooling1D()           average them across the passage
Dense(7, activation="softmax")     one probability per author
```

The averaging step throws word order away completely. What the model learns is
**which words an author reaches for and how often** — which makes it, for all
the neural machinery, a learned bag of words. That is the same signal Burrows's
Delta uses in module 9, arrived at by a different route.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# The corpus, as the project described it.
CASE_CORPUS = [
    ("Amos Tutuola",      10,   474_474),
    ("Ben Okri",          11,   780_526),
    ("Chinua Achebe",      7,   369_803),
    ("Buchi Emecheta",     3,   188_866),
    ("Aminatta Forna",     5,   662_698),
    ("Nadine Gordimer",   17, 1_737_979),
    ("Ngugi wa Thiong'o",  7,   812_065),
    ("Wole Soyinka",       1,    91_226),
]
print(f"{'author':<22}{'novels':>8}{'words':>12}")
print("-" * 42)
for a, n, w in CASE_CORPUS:
    print(f"{a:<22}{n:>8}{w:>12,}")
tot_b = sum(n for _, n, _ in CASE_CORPUS)
tot_w = sum(w for _, _, w in CASE_CORPUS)
print("-" * 42)
print(f"{'TOTAL':<22}{tot_b:>8}{tot_w:>12,}")
print()
print("Two decisions follow immediately from this table, and both are")
print("methodological rather than technical:")
print("  1. Soyinka has ONE novel. He cannot be both trained on and tested on,")
print("     so he is dropped. Seven authors remain.")
print("  2. Gordimer has a third of the words. Left unbalanced, a model can")
print("     score well by leaning on 'probably Gordimer'.")

In [ ]:
# Balancing, and what it costs. Emecheta is the binding constraint.
CHUNK = 400
per_author = {a: w // CHUNK for a, n, w in CASE_CORPUS if n >= 2}
print("passages available at 400 words each:\n")
for a, p in sorted(per_author.items(), key=lambda kv: kv[1]):
    print(f"   {a:<22}{p:>6,}")
floor = min(per_author.values())
print()
print(f"Balanced, every author is cut to the smallest: {floor:,} passages.")
print(f"That discards {sum(per_author.values()) - floor * len(per_author):,} "
      f"of {sum(per_author.values()):,} available passages.")
print()
print("This is the usual trade. Balance protects against a lazy majority-class")
print("model; it also throws away most of your data. Say which you chose.")

### 🔧 Your turn 1

Open the repository at https://github.com/Sagaustus/cnn_dh_practice and read `notebooks/corpus.py`. Find the line that drops single-book authors. Would you have made the same call?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The alternative is to keep Soyinka and accept that his score is meaningless, or
to merge him into an "other" class — which changes the research question from
"who wrote this" to "did one of these seven write this". Neither is wrong; both
need stating. The general rule this case illustrates is that the *shape of your
corpus* determines your research question more than your choice of model does,
and a methods section that describes the architecture in detail while passing
over the corpus in a sentence has its priorities backwards.

</details>